In [2]:
import pandas as pd
import duckdb

In [3]:
df = pd.read_csv("online_retail.csv" , parse_dates=["InvoiceDate"]) 

# # Create an in-memory DuckDB connection.
conn = duckdb.connect()
conn.register("retail" , df)

In [4]:
conn.execute("SELECT COUNT(*) FROM retail").fetchdf()
conn.execute("SELECT * FROM retail LIMIT 5").df()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
#pandas work
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
result = df.groupby("Country")["Revenue"].sum().sort_values(ascending = False)
result.head()

Country
United Kingdom    8187806.364
Netherlands        284661.540
EIRE               263276.820
Germany            221698.210
France             197403.900
Name: Revenue, dtype: float64

In [6]:
# duckdb
duck_result = conn.execute("""
SELECT Country , SUM(Quantity * UnitPrice) AS Revenue
FROM retail
GROUP BY Country
ORDER BY Revenue DESC
""").fetchdf()
duck_result.head()

,Country,Revenue
0,United Kingdom,8.187806e+06
1,Netherlands,2.846615e+05
2,EIRE,2.632768e+05
3,Germany,2.216982e+05
4,France,1.974039e+05


In [7]:
df[df['Quantity'] < 0].shape[0]

10624

In [8]:
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df[df['Quantity'] < 0]['InvoiceNo'].str.startswith('C').value_counts()

InvoiceNo
True     9288
False    1336
Name: count, dtype: int64

In [9]:
df[df["Quantity"] < 0 ]["CustomerID"].isna().sum()

np.int64(1719)

In [10]:
conn.execute("""
    CREATE OR REPLACE VIEW retail_clean AS
    SELECT *,
           Quantity * UnitPrice AS Revenue
    FROM retail
    WHERE CustomerID IS NOT NULL
""")

conn.execute("SELECT COUNT(*) FROM retail_clean").fetchdf()

,count_star()
0,406829


In [11]:
### cohort analysis

In [12]:
conn.execute(""" 
    WITH cohorts AS (
        SELECT 
            CustomerID,
            DATE_TRUNC("month" , MIN(InvoiceDate)) AS cohort_month
        FROM retail_clean
        GROUP BY CustomerID
""")
conn.execute("SELECT * FROM cohorts LIMIT 5").fetchdf()

ParserException: Parser Error: syntax error at end of input

SyntaxError: invalid syntax (706574846.py, line 1)